In [4]:
import pandas as pd
import numpy as np

np.random.seed(42)
num_rows = 100

age = np.random.randint(18, 65, size=num_rows)
city = np.random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix'], size=num_rows)
salary = np.random.randint(45000, 120000, size=num_rows)
user_id = np.arange(1, num_rows + 1)


city_weights = {
    'New York': 15,
    'Los Angeles': 10,
    'Chicago': 5,
    'Houston': 0,
    'Phoenix': -5
}

city_bonus = pd.Series(city).map(city_weights).values

# Define the formula: Score is influenced by age, salary, and city
# We scale salary by dividing by 1000 to keep it from dominating the equation
base_score = (age * 0.5) + (salary / 1000 * 0.3) + city_bonus

noise = np.random.normal(loc=0, scale=5, size=num_rows) # Gaussian noise with mean=0, std=5
final_score = base_score + noise

# Ensure the score stays within a logical range (e.g., 1 to 100) and round it
final_score = np.clip(final_score, 1, 100)
final_score = np.round(final_score, 2)

df = pd.DataFrame({
    'UserID': user_id,
    'Age': age,
    'City': city,
    'Salary': salary,
    'Score': final_score
})

# --- 4. Display and Save ---
print("Generated DataFrame Head (with correlated Score):")
print(df.head())

# Save the DataFrame to a new CSV file
output_path = "Data/simple_dataset.csv"
df.to_csv(output_path, index=False)

print(f"\nSuccessfully saved dataset with correlated score to '{output_path}'")

Generated DataFrame Head (with correlated Score):
   UserID  Age      City  Salary  Score
0       1   56   Chicago   91717  55.16
1       2   46   Chicago   95859  59.17
2       3   32  New York   71309  51.28
3       4   60   Chicago  108734  71.19
4       5   25   Chicago  115467  54.51

Successfully saved dataset with correlated score to 'Data/simple_dataset.csv'


In [1]:
import pandas as pd

file_path = "Data/simple_dataset.csv"

df = pd.read_csv(file_path)

print(df.head())

   UserID  Age      City  Salary  Score
0       1   56   Chicago   91717  55.16
1       2   46   Chicago   95859  59.17
2       3   32  New York   71309  51.28
3       4   60   Chicago  108734  71.19
4       5   25   Chicago  115467  54.51


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

df_processed = pd.get_dummies(df, columns=['City'], drop_first=True)

X = df_processed.drop(['UserID', 'Score'], axis=1)
y = df_processed['Score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=674)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)


mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("--- Model Performance ---")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R²): {r2:.2f}")


print("\n--- Model Coefficients ---")

coeffs = pd.DataFrame(model.coef_, X_train.columns, columns=['Coefficient'])
print(coeffs)


--- Model Performance ---
Mean Squared Error (MSE): 29.99
R-squared (R²): 0.75

--- Model Coefficients ---
                  Coefficient
Age                  0.564729
Salary               0.000333
City_Houston        -2.968734
City_Los Angeles     8.420214
City_New York       12.329229
City_Phoenix        -8.096262
